## On the negative NPMI for Top2Vec (-0.17)

Top2Vec produced topics whose top-word NPMI score on this corpus is **negative**
(roughly -0.17 in our run), meaning the top words selected for each topic **co-occur
less often than chance** in the document collection. This is an **anti-coherence**
signal, not just "weak" coherence, and we therefore **exclude Top2Vec from the
recommended unsupervised configuration** for this dataset.

Why this happens here:

- Top2Vec relies on `doc2vec` (or, optionally, a sentence-transformer) embeddings of
  full documents, then clusters with HDBSCAN. With ~3.9k short, often near-duplicate
  patient-provider exchanges, doc2vec embeddings are underdetermined and HDBSCAN tends
  to produce one large noise cluster plus a few small topics whose representative
  words generalize poorly across documents.
- BERTopic, by contrast, uses pretrained sentence-transformer embeddings + UMAP +
  HDBSCAN + class-based TF-IDF, which is far more robust at this corpus size and
  produces positive NPMI on the same data.

**Reported in the paper:** Top2Vec is included in Table X for completeness, but
flagged with the negative-NPMI caveat and not carried into the recommendations.


## Input corpus: shared training split

To keep the unsupervised topic models (BERTopic, Top2Vec, LDA) directly comparable
with the supervised pipeline, we fit them on the **training portion of the shared
split** produced by `create_shared_split.py` (dedup'd, restricted to labels with
>= 100 occurrences in the pooled corpus). The next cell reconstructs that input.
Replace the loose `df_diabetes` / `df_covid` reads further down with `train_docs`.


In [ ]:
# SHARED_SPLIT_TOPIC_INPUT_v1
# Build the training-split corpus that the supervised models see, for fair comparison.
import random, numpy as np, pandas as pd

SEED = 42
random.seed(SEED); np.random.seed(SEED)

dfCovid    = pd.read_csv("data/dataRwaCovid.csv", on_bad_lines="skip", low_memory=False)
dfDiabetes = pd.read_csv("data/dataIHADiabetes.csv", on_bad_lines="skip", low_memory=False)

DiabetesTopics = ["Physical","Psychological","No Symptoms","Prognosis","Laboratory/Testing","Imaging","Clinical","Testing/Monitoring Devices","Health Data","Diagnostic Methods - Other","Medications","Procedures","Alternative","Physical Therapy","Counseling","Adverse Events","Therapeutic Devices","Treatment(Rx) - Other","Outpatient Logistics/Scheduling","Hospitalizations","Insurance/Billing","Medical Records","Referrals","Transportation","Primary (Pharmaceutical Prevention)","Primary (Non-Pharmaceutical Prevention)","Secondary (Pharmaceutical Prevention)","Secondary (Non-Pharmaceutical Prevention)","Diet/Nutrition","Exercise","Substance Use","Entertainment","Lifestyle - Other","Housing","Work/School","Social Services","Friends & Family","Cultural/Religion","Travel","Physical Environment/Climate","Financial","Social - Other","Technical/IT","Safety Concerns","Health Education","Sexual & Reproductive Health","Child & Family Health","Problems Solved","Grateful Patient","Service Complaint","Request to Stop","Emergent","Urgent","Non-urgent","Stigma Present","Rapport","Transition to Adult Clinic"]
CovidTopics = ["Physical","Mental/Emotional","No Symptoms","Laboratory/Testing","Imaging","Clinical","Diagnostic Methods - Other","Medications","Procedures","Alternative","Physical Therapy","Counseling","Treatment(Rx) - Other","Outpatient Logistics/Scheduling","Hospitalizations","Pharmaceutical Prevention","Non-Pharmaceutical Prevention","Diet/Nutrition","Exercise","Substance Use","Lifestyle - Other","Housing","Work/School","Social Services","Friends & Family","Cultural/Religion","Travel","Physical Environment/Climate","Financial","Social - Other","Technical/IT","Safety concern","Health Education","Maternal & Child Health","Problems Solved","Grateful Patient","Service Complaint","Request to Stop","Emergent","Urgent","Non-urgent","Stigma Present","wave","batch"]
common = sorted(set(DiabetesTopics) & set(CovidTopics))

dfD = dfDiabetes[["conversation"] + common].copy(); dfD["source"] = "Canada"
dfC = dfCovid[["conversation(english_only)"] + common].rename(
    columns={"conversation(english_only)": "conversation"}).copy(); dfC["source"] = "Rwanda"

dfCombined = pd.concat([dfC, dfD]).reset_index(drop=True)
label_keep = dfCombined[common].sum()[lambda s: s >= 100].index.tolist()
dfFilt = (dfCombined[["conversation", "source"] + label_keep]
          .drop_duplicates(subset="conversation")
          .reset_index(drop=True))

_split = np.load("data/shared_split_indices.npz")
train_idx = _split["train_idx"]

train_docs    = dfFilt.iloc[train_idx]["conversation"].tolist()
train_sources = dfFilt.iloc[train_idx]["source"].tolist()
print(f"Training docs for unsupervised models: {len(train_docs)} "
      f"(Rwanda {sum(s=='Rwanda' for s in train_sources)}, "
      f"Canada {sum(s=='Canada' for s in train_sources)})")
print(f"Labels kept (only used for downstream comparison): {len(label_keep)}")


In [ ]:
!pip install bertopic

In [ ]:
import pandas as pd
from bertopic import BERTopic

In [ ]:
df_diabetes = pd.read_csv("dataIHADiabetes.csv")
df_covid    = pd.read_csv("cleaned_and_extracted_dataset-2.csv")

In [ ]:
df_diabetes = df_diabetes["conversation"]


In [ ]:
df_diabetes.head()


,conversation
0,(S) : Hi! Welcome to WelTel's Interior Health ...
1,"(S) : If you're finding lumps, try to find alt..."
2,(S) : DATA REVIEW followed by a text response ...
3,"(S) : ""I may have diabetes BUT diabetes does n..."
4,(S) : Time spent amongst treesðŸŒ³ðŸŒ³ is neve...


In [ ]:

df_covid= df_covid.rename(columns={'conversation(english_only)': 'conversation'})

In [ ]:
df_covid = df_covid["conversation"]

In [ ]:
df_covid.head()

,conversation
0,"(S): Hello, how are you today? Do you have any..."
1,(S): Welcome to Rwanda Ministry of Health plat...
2,(S): Welcome to Rwanda Ministry of Health plat...
3,"(S): Hello, how are you today? Do you have any..."
4,(S): Welcome to Rwanda Ministry of Health plat...


In [ ]:
df = pd.concat([df_diabetes, df_covid])

In [ ]:
df.head()

,conversation
0,(S) : Hi! Welcome to WelTel's Interior Health ...
1,"(S) : If you're finding lumps, try to find alt..."
2,(S) : DATA REVIEW followed by a text response ...
3,"(S) : ""I may have diabetes BUT diabetes does n..."
4,(S) : Time spent amongst treesðŸŒ³ðŸŒ³ is neve...


In [ ]:
import pandas as pd
import numpy as np
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance, PartOfSpeech
from sklearn.feature_extraction.text import CountVectorizer
from hdbscan import HDBSCAN

docs = df
hdbscan_model = HDBSCAN(min_cluster_size=30, metric='euclidean', cluster_selection_method='eom', prediction_data=True)

keybert_model = KeyBERTInspired(top_n_words=6)
mmr_model = MaximalMarginalRelevance(diversity=0.5,top_n_words=6)
pos_model = PartOfSpeech("en_core_web_sm",top_n_words=6)

representation_model = {
    "Main": keybert_model,
    "Diverse_Keywords": mmr_model,
    "Nouns_and_Adjectives": pos_model
}

vectorizer_model = CountVectorizer(stop_words="english")

topic_model = BERTopic(
    language="english",
    hdbscan_model=hdbscan_model,
    representation_model=representation_model,
    vectorizer_model=vectorizer_model,
    top_n_words=6,
    verbose=True
)
topics, _ = topic_model.fit_transform(docs)
print("ITS WORKING:",topic_model.get_topic_info(),"YES IT DOES")
topic_info = topic_model.get_topic_info()
topic_info = topic_info.set_index('Topic')

if 'Main' in topic_info.columns:
    name_column = 'Main'
else:
    name_column = 'Name'

topic_distr, _ = topic_model.approximate_distribution(docs)

probability_threshold = 0.05
multi_label_topics_ids = []
multi_label_topics_names = []
for doc_distr in topic_distr:
    relevant_topic_ids_array = np.where(doc_distr > probability_threshold)[0]

    multi_label_topics_ids.append(relevant_topic_ids_array)

    if relevant_topic_ids_array.size > 0:
        doc_topic_names = topic_info.loc[relevant_topic_ids_array][name_column].tolist()
    else:
        doc_topic_names =[]
    multi_label_topics_names.append(doc_topic_names)

results_df = pd.DataFrame({
    'Document': docs,
    'Dominant_Topic': topics,
    'Multi_Label_Topic_IDs': multi_label_topics_ids,
    'Multi_Label_Topic_Names': multi_label_topics_names
})
print(results_df)

2025-08-17 06:05:25,504 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/123 [00:00<?, ?it/s]

2025-08-17 06:05:39,078 - BERTopic - Embedding - Completed ✓
2025-08-17 06:05:39,081 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-08-17 06:06:31,554 - BERTopic - Dimensionality - Completed ✓
2025-08-17 06:06:31,555 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-08-17 06:06:31,683 - BERTopic - Cluster - Completed ✓
2025-08-17 06:06:31,688 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-08-17 06:06:38,419 - BERTopic - Representation - Completed ✓


ITS WORKING:     Topic  Count                                            Name  \
0      -1    611          -1_coronavirus_rwanda_symptom_symptoms   
1       0   1173                   0_hello_covid19_covid_texting   
2       1    731               1_covid_covid19_illness_telephone   
3       2    248                 2_diabetes_holidays_26th_clinic   
4       3    174              3_mwaramutse_covid_covid19_illness   
5       4    173              4_coronavirus_symptom_fever_rwanda   
6       5    138               5_holidays_happy_emergency_spring   
7       6    110        6_inactivated_patient_reactivated_health   
8       7     72         7_coronavirus_greetings_covid19_symptom   
9       8     70               8_coronavirus_hello_symptom_today   
10      9     56              9_coronavirus_rwanda_fever_symptom   
11     10     52       10_diabetes_confident_questions_answering   
12     11     45        11_inactivated_reactivated_weltel_active   
13     12     42  12_hypoglycemia_h

100%|██████████| 4/4 [00:02<00:00,  1.34it/s]


                                               Document  Dominant_Topic  \
0     (S) : Hi! Welcome to WelTel's Interior Health ...              -1   
1     (S) : If you're finding lumps, try to find alt...              10   
2     (S) : DATA REVIEW followed by a text response ...               2   
3     (S) : "I may have diabetes BUT diabetes does n...              -1   
4     (S) : Time spent amongst treesðŸŒ³ðŸŒ³ is neve...              -1   
...                                                 ...             ...   
2786  (S): haven't heard from you yet. how are you? ...              -1   
2787  (C): Hello #Perpetue#! How are you? Were you t...               1   
2788  (S): Hello, this is the texting line used for ...               0   
2789  (C): Good morning! This is Babyl Health Rwanda...               1   
2790  (S): Hello, do you have any health or COVID re...               1   

        Multi_Label_Topic_IDs  \
0        [0, 1, 2, 5, 13, 15]   
1              [2, 5, 10, 18]   


In [ ]:
probability_threshold = 0.09
multi_label_topics_ids = []
multi_label_topics_names = []
for doc_distr in topic_distr:
    relevant_topic_ids_array = np.where(doc_distr > probability_threshold)[0]

    multi_label_topics_ids.append(relevant_topic_ids_array)

    if relevant_topic_ids_array.size > 0:
        doc_topic_names = topic_info.loc[relevant_topic_ids_array][name_column].tolist()
    else:
        doc_topic_names =[]
    multi_label_topics_names.append(doc_topic_names)

results_df = pd.DataFrame({
    'Topic_IDs': multi_label_topics_ids,
    'Topic_Names': multi_label_topics_names
})
print(results_df)

              Topic_IDs                                        Topic_Names
0            [2, 5, 13]  [2_diabetes_holidays_26th_clinic, 5_holidays_h...
1        [2, 5, 10, 18]  [2_diabetes_holidays_26th_clinic, 5_holidays_h...
2       [2, 10, 12, 18]  [2_diabetes_holidays_26th_clinic, 10_diabetes_...
3           [2, 10, 18]  [2_diabetes_holidays_26th_clinic, 10_diabetes_...
4            [2, 5, 10]  [2_diabetes_holidays_26th_clinic, 5_holidays_h...
...                 ...                                                ...
3902      [1, 6, 9, 11]  [1_covid_covid19_illness_telephone, 6_inactiva...
3903  [1, 3, 6, 11, 14]  [1_covid_covid19_illness_telephone, 3_mwaramut...
3904       [0, 1, 7, 8]  [0_hello_covid19_covid_texting, 1_covid_covid1...
3905      [1, 3, 6, 11]  [1_covid_covid19_illness_telephone, 3_mwaramut...
3906          [0, 1, 3]  [0_hello_covid19_covid_texting, 1_covid_covid1...

[3907 rows x 2 columns]


In [ ]:
!pip install top2vec
from top2vec import Top2Vec

if isinstance(docs, (pd.Series, pd.DataFrame)):
    docs_list = docs.tolist()
else:
    docs_list = list(docs)

top2vec_model = Top2Vec(documents=docs_list, embedding_model='universal-sentence-encoder')

num_topics = top2vec_model.get_num_topics()

topic_words, word_scores, topic_nums = top2vec_model.get_topics(num_topics)

num_words_to_display = 6

print(f"Top 6 Words for Each Topic")

for i in range(17):
  topic_num = topic_nums[i]
  top_words_for_topic = topic_words[i][:num_words_to_display]
  word_scores_for_topic = word_scores[i][:num_words_to_display]
  print(word_scores_for_topic)

  print(f"\nTopic {topic_num}:")
  print(' '.join(top_words_for_topic))


2025-08-17 06:06:53,179 - top2vec - INFO - Pre-processing documents for training
INFO:top2vec:Pre-processing documents for training
2025-08-17 06:06:54,795 - top2vec - INFO - Downloading universal-sentence-encoder model
INFO:top2vec:Downloading universal-sentence-encoder model
2025-08-17 06:07:44,905 - top2vec - INFO - Creating joint document/word embedding
INFO:top2vec:Creating joint document/word embedding
2025-08-17 06:07:50,496 - top2vec - INFO - Creating lower dimension embedding of documents
INFO:top2vec:Creating lower dimension embedding of documents
2025-08-17 06:08:12,516 - top2vec - INFO - Finding dense areas of documents
INFO:top2vec:Finding dense areas of documents
2025-08-17 06:08:12,601 - top2vec - INFO - Finding topics
INFO:top2vec:Finding topics


Top 6 Words for Each Topic
[0.28693154 0.27057695 0.25272065 0.25185287 0.24175093 0.22627215]

Topic 0:
messages greetings texting hi hello reply
[0.21896195 0.21772696 0.21056734 0.20288265 0.19306508 0.19180605]

Topic 1:
hotline illness health patient tested re
[0.3427351  0.34238896 0.2562301  0.24375933 0.20574446 0.19535023]

Topic 2:
symptom symptoms illness flu re health
[0.3014471  0.24877934 0.2376104  0.2293275  0.21377304 0.20875686]

Topic 3:
hotline telephone phone sms texting messages
[0.31775886 0.28874552 0.27649024 0.26873857 0.24307445 0.21598089]

Topic 4:
symptoms hi greetings hello symptom morning
[0.32092032 0.29953766 0.2789727  0.2656779  0.22606246 0.20958272]

Topic 5:
hi greetings hello merry fine welcome
[0.21683423 0.20692323 0.1954192  0.18838958 0.18616742 0.18570375]

Topic 6:
dexcom appointment sugars insulin diabetes healthy
[0.3045201  0.23311168 0.22014213 0.21730745 0.20936038 0.20750786]

Topic 7:
confident hi greetings survey diabetes hello
[0.3

In [ ]:
results_df.to_csv('BerTopic_results.csv', index=False)

In [ ]:
!pip install top2vec
!pip install gensim
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import pandas as pd

nltk.download('stopwords')
nltk.download('wordnet')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    text = re.sub(r'[^a-zA-Z\s]', '', text, re.I|re.A)
    text = text.lower()
    text = text.strip()

    tokens = []
    for token in text.split():
        if token not in stop_words:
            tokens.append(lemmatizer.lemmatize(token))

    return " ".join(tokens)


preprocessed_docs = [preprocess_text(doc) for doc in docs]

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation


vectorizer_lda = CountVectorizer(max_df=0.9, min_df=5, ngram_range=(1,2))
X_lda = vectorizer_lda.fit_transform(preprocessed_docs)


lda_model = LatentDirichletAllocation(n_components=17, random_state=42)
lda_model.fit(X_lda)

def display_topics(model, feature_names, num_top_words):
    topics = []
    for topic_idx, topic in enumerate(model.components_):
        topic_words = [feature_names[i] for i in topic.argsort()[:-num_top_words - 1:-1]]
        print(f"Topic {topic_idx}: {' '.join(topic_words)}")
        topics.append(topic_words)
    return topics

print("LDA Topics")
feature_names_lda = vectorizer_lda.get_feature_names_out()
lda_topics = display_topics(lda_model, feature_names_lda, 10)

LDA Topics
Topic 0: dr dr eiko eiko good im week great work know happy
Topic 1: thank good covid morning call situation telephone good morning know know situation
Topic 2: share share today health covid health like would would like today concern would like share
Topic 3: covid hello good patient care home based care patient home hotline following following
Topic 4: survey dr eiko dr eiko get good short would thank reminder
Topic 5: go contact patient patient inactivated inactivated tomorrow testing go health didnt facility
Topic 6: every please checking would two dr good like two week week
Topic 7: diabetes text texting month tir helping good screenshot project question
Topic 8: test tested negative tested negative get day result negative covid ok want
Topic 9: confident dr eiko dr eiko diabetes thanks good scale diabetes confident question
Topic 10: great week much good thank pumpkin helpful response texting im
Topic 11: dr good dr eiko eiko im insulin hear ok im good get
Topic 12: co

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF


vectorizer_nmf = TfidfVectorizer(max_df=0.9, min_df=5, ngram_range=(1,2))
X_nmf = vectorizer_nmf.fit_transform(preprocessed_docs)

nmf_model = NMF(n_components=17, random_state=42)
nmf_model.fit(X_nmf)

print("\nNMF Topics")
feature_names_nmf = vectorizer_nmf.get_feature_names_out()
nmf_topics = display_topics(nmf_model, feature_names_nmf, 10)


NMF Topics
Topic 0: reaction line used used covid hello texting texting line sent daily reaction today concern message basis reaction daily basis
Topic 1: new symptom yesno yesno tell ok yesno tell new symptom ok new tell symptom ok
Topic 2: go patient inactivated inactivated health facility facility contact didnt shes advised testing shes didnt go facility testing
Topic 3: symptom coronavirus today symptom hello today coronavirus symptom coronavirus hello hello today cooperation thank cooperation
Topic 4: call telephone phone situation know situation want know patient covid based patient know want
Topic 5: patient home following patient care covid mwaramutse hotline following uyu based care rugo hello uyu ni mugukurikirana
Topic 6: dr eiko dr eiko im great good week im good hear ok
Topic 7: share today health covid share concern would like share hello health would like like would health
Topic 8: administrator active weltel please active weltel longer active please contact contact adm

In [ ]:
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary
from gensim.utils import simple_preprocess

tokenized_docs = [doc.split() for doc in preprocessed_docs]

gensim_dictionary = Dictionary(tokenized_docs)

def calculate_coherence(topics, tokenized_docs, dictionary):

    coherence_model = CoherenceModel(
        topics=topics,
        texts=tokenized_docs,
        dictionary=dictionary,
        coherence='c_v'
    )
    return coherence_model.get_coherence()
def calculate_coherence_npmi(topics, tokenized_docs, dictionary):

    coherence_model = CoherenceModel(
        topics=topics,
        texts=tokenized_docs,
        dictionary=dictionary,
        coherence='c_npmi'
    )
    return coherence_model.get_coherence()

analyzer_lda = vectorizer_lda.build_analyzer()
tokenized_docs_lda = [analyzer_lda(doc) for doc in preprocessed_docs]
gensim_dictionary_lda = Dictionary(tokenized_docs_lda)
lda_topics = [t for t in lda_topics if t]
coherence_lda = calculate_coherence(lda_topics, tokenized_docs_lda, gensim_dictionary_lda)
coherence_lda_npmi = calculate_coherence_npmi(lda_topics, tokenized_docs_lda, gensim_dictionary_lda)
print(f"LDA Coherence CV Score: {coherence_lda:.4f}")
print(f"LDA coherence NPMI Score:{coherence_lda_npmi:.4f}")


analyzer_nmf = vectorizer_nmf.build_analyzer()
tokenized_docs_nmf = [analyzer_nmf(doc) for doc in preprocessed_docs]
gensim_dictionary_nmf = Dictionary(tokenized_docs_nmf)
nmf_topics = [t for t in nmf_topics if t]
coherence_nmf = calculate_coherence(nmf_topics, tokenized_docs_nmf, gensim_dictionary_nmf)
coherence_nmf_npmi = calculate_coherence_npmi(nmf_topics, tokenized_docs_nmf, gensim_dictionary_nmf)
print(f"NMF Coherence CV Score: {coherence_nmf:.4f}")
print(f"NMF Coherence NPMI Score: {coherence_nmf_npmi:.4f}")



bertopic_info = topic_model.get_topic_info()
bertopic_topics = []
for topic_num in bertopic_info.Topic:
    if topic_num != -1:
        # print(f"Topic {topic_num}:")

        words = [word for word, _ in topic_model.get_topic(topic_num)[:10]]
        # print(len(topic_model.get_topic(topic_num)))
        # print(' '.join(words))
        bertopic_topics.append(words)
bertopic_topics = [t for t in bertopic_topics if t]
coherence_bertopic = calculate_coherence(bertopic_topics, tokenized_docs_lda, gensim_dictionary_lda)
coherence_bertopic_npmi = calculate_coherence_npmi(bertopic_topics, tokenized_docs_lda, gensim_dictionary_lda)
print(f"BERTopic Coherence Score: {coherence_bertopic:.4f}")
print(f"BERTopic Coherence NPMI Score: {coherence_bertopic_npmi:.4f}")

top2vec_topics = [list(words)[:10] for words in top2vec_model.topic_words]
top2vec_topics = [[word for word in t] for t in top2vec_topics]
top2vec_topics = [t for t in top2vec_topics if t]
coherence_top2vec = calculate_coherence(top2vec_topics, tokenized_docs_lda, gensim_dictionary_lda)
coherence_top2vec_npmi = calculate_coherence_npmi(top2vec_topics, tokenized_docs_lda, gensim_dictionary_lda)
print(f"Top2Vec Coherence Score: {coherence_top2vec:.4f}")
print(f"Top2Vec Coherence NPMI Score: {coherence_top2vec_npmi:.4f}")






def calculate_topic_diversity(topics):

    num_top_words = len(topics[0])

    all_words = [word for topic in topics for word in topic]

    unique_words = set(all_words)

    diversity = len(unique_words) / len(all_words)
    return diversity

diversity_lda = calculate_topic_diversity(lda_topics)
diversity_nmf = calculate_topic_diversity(nmf_topics)
diversity_bertopic = calculate_topic_diversity(bertopic_topics)
diversity_top2vec = calculate_topic_diversity(top2vec_topics)


print("2. Topic Diversity Scores")
print(f"   LDA Diversity Score:      {diversity_lda:.4f}")
print(f"   NMF Diversity Score:      {diversity_nmf:.4f}")
print(f"   BERTopic Diversity Score: {diversity_bertopic:.4f}")
print(f"   Top2Vec Diversity Score:  {diversity_top2vec:.4f}")




LDA Coherence CV Score: 0.7117
LDA coherence NPMI Score:0.0847
NMF Coherence CV Score: 0.8423
NMF Coherence NPMI Score: 0.0833
BERTopic Coherence Score: 0.7253
BERTopic Coherence NPMI Score: 0.1382
Top2Vec Coherence Score: 0.3414
Top2Vec Coherence NPMI Score: -0.1722
2. Topic Diversity Scores
   LDA Diversity Score:      0.6647
   NMF Diversity Score:      0.9765
   BERTopic Diversity Score: 0.6140
   Top2Vec Diversity Score:  0.2741
